In [2]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("SahlPay-Partie2-RDD")
    .master("spark://spark-master:7077")
    .config("spark.driver.host", "jupyter")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.executor.memory", "1g")
    .config("spark.jars.ivy", "/tmp/.ivy2")
    .config("spark.jars.packages",
            "org.apache.hadoop:hadoop-aws:3.3.4,"
            "com.amazonaws:aws-java-sdk-bundle:1.12.262")
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000")
    .config("spark.hadoop.fs.s3a.access.key", "sahlpay")
    .config("spark.hadoop.fs.s3a.secret.key", "sahlpay123")
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .getOrCreate()
)
sc = spark.sparkContext
print(spark.version, sc.master, sc.defaultParallelism)

3.5.5 spark://spark-master:7077 4


In [3]:
import json

lines = sc.textFile("s3a://lake/bronze/history/transactions_history.jsonl")
print("Partitions :", lines.getNumPartitions())

def parse(line):
    try:
        return json.loads(line)
    except Exception:
        return None

tx = lines.map(parse).filter(lambda t: t is not None)
print("Nb lignes   :", lines.count())
print("Nb parsées  :", tx.count())
print(tx.first())

26/11/18 23:20:12 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


Partitions : 2


Nb lignes   : 50000
Nb parsées  : 50000
{'tx_id': 'TX-00016455', 'type': 'P2M', 'payer_wallet': 'W-10158', 'payee_id': 'M-046', 'device_id': 'D-370bc0', 'city': 'Fes', 'channel': 'QR', 'status': 'INITIATED', 'amount_mad': 104.08, 'event_time': '2026-11-15T23:02:16Z'}


In [4]:
completed = tx.filter(lambda t: t["status"] == "COMPLETED")
completed.cache()
print("COMPLETED :", completed.count())

[Stage 3:>                                                          (0 + 2) / 2]

COMPLETED : 15525


In [5]:
import time
t0 = time.time()

total_city = (completed
              .map(lambda t: (t["city"], t["amount_mad"]))
              .reduceByKey(lambda a, b: a + b))

res_total = total_city.collect()
t_total = time.time() - t0

for city, amount in sorted(res_total, key=lambda x: -x[1]):
    print(f"{city:15s} {amount:15,.2f} MAD")
print(f"Temps : {t_total:.2f} s")

Casablanca           899,587.45 MAD
Rabat                582,048.63 MAD
Marrakech            454,522.39 MAD
Fes                  242,935.22 MAD
Tanger               232,327.27 MAD
Agadir               180,523.61 MAD
Kenitra              133,977.89 MAD
Oujda                128,017.24 MAD
Meknes               117,299.19 MAD
Tetouan              104,792.05 MAD
Temps : 1.21 s


In [6]:
t0 = time.time()

sum_count = (completed
             .map(lambda t: (t["city"], (t["amount_mad"], 1)))
             .reduceByKey(lambda a, b: (a[0] + b[0], a[1] + b[1])))

avg_city = sum_count.mapValues(lambda v: (v[0], v[1], v[0] / v[1]))
res_avg = avg_city.collect()
t_avg = time.time() - t0

print(f"{'Ville':15s} {'Total':>15s} {'Nb':>8s} {'Moyenne':>10s}")
for city, (s, n, m) in sorted(res_avg, key=lambda x: -x[1][0]):
    print(f"{city:15s} {s:15,.2f} {n:8d} {m:10.2f}")
print(f"Temps : {t_avg:.2f} s")

Ville                     Total       Nb    Moyenne
Casablanca           899,587.45     4367     206.00
Rabat                582,048.63     2893     201.19
Marrakech            454,522.39     2041     222.70
Fes                  242,935.22     1336     181.84
Tanger               232,327.27     1141     203.62
Agadir               180,523.61     1101     163.96
Kenitra              133,977.89      709     188.97
Oujda                128,017.24      691     185.26
Meknes               117,299.19      676     173.52
Tetouan              104,792.05      570     183.85
Temps : 0.43 s


In [7]:
from operator import add

top5 = (completed
        .filter(lambda t: t["type"] == "P2M")
        .map(lambda t: (t["payee_id"], 1))
        .reduceByKey(add)
        .takeOrdered(5, key=lambda x: -x[1]))

for rank, (m, n) in enumerate(top5, 1):
    print(rank, m, n)

1 M-124 59
2 M-237 58
3 M-055 57
4 M-240 55
5 M-116 55


In [8]:
print(avg_city.toDebugString().decode("utf-8"))

(2) PythonRDD[16] at collect at /tmp/ipykernel_327/535929726.py:8 []
 |  MapPartitionsRDD[15] at mapPartitions at PythonRDD.scala:160 []
 |  ShuffledRDD[14] at partitionBy at <unknown>:0 []
 +-(2) PairwiseRDD[13] at reduceByKey at /tmp/ipykernel_327/535929726.py:3 []
    |  PythonRDD[12] at reduceByKey at /tmp/ipykernel_327/535929726.py:3 []
    |  PythonRDD[5] at RDD at PythonRDD.scala:53 []
    |      CachedPartitions: 2; MemorySize: 752.8 KiB; DiskSize: 0.0 B
    |  s3a://lake/bronze/history/transactions_history.jsonl MapPartitionsRDD[1] at textFile at <unknown>:0 []
    |  s3a://lake/bronze/history/transactions_history.jsonl HadoopRDD[0] at textFile at <unknown>:0 []
